# Computational Comparison of Three English Translations of Chekhov's *"The Lady with the Dog"*

**Research question:** If a translation is a model of the original text, do different translations of the same story produce measurably different models?

## Table of Contents

* [1. Imports](#setup)
* [2. Collecting data](#data)
* [3. NLP Pipeline: Tokenization and Lemmatization](#pipeline)
* [4. Lexical Richness: Type-Token Ratio (TTR)](#ttr)
* [5. Frequency Analysis: Top 20 Lemmas](#frequency)
* [6. Key Motif Frequencies](#motifs)
* [7. KWIC Concordances](#kwic)
* [8. Sentiment Trajectory](#sentiment)
* [9. Character Analysis: Focalization](#character)
* [10. Hapax Legomena](#hapax)
* [11. Syntactic Profile: Sentence Length and POS Density](#syntax)
* [12. Discussion](#discussion)
* [13. Conclusion](#conclusion)

## 1. Imports <a class="anchor" id="setup"></a>
[Back to Table of Contents](#table-of-contents)

In [138]:
import re
import urllib.request
from collections import Counter
import math

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

import io
from pypdf import PdfReader
import requests

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.text import Text
from nltk.sentiment.vader import SentimentIntensityAnalyzer

STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()
ANALYZER   = SentimentIntensityAnalyzer()

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /Users/ssonyii/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ssonyii/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/ssonyii/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /Users/ssonyii/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/ssonyii/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/ssonyii/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/ssonyii/nltk_data.

True

In [ ]:
PALETTE = {
    'Garnett (1917)':'#6495ED',
    'Koteliansky & Cannon (1917)': '#890835',
    'Litvinov (1973)': '#efa942',
}

## 2. Collecting data <a class="anchor" id="data"></a>
[Back to Table of Contents](#table-of-contents)

### 2.1 Fetching from Project Gutenberg

Two translations are available as plain-text files on Project Gutenberg. The cell below fetches each file from the URL.

In [39]:
SOURCES = {
    'Garnett (1917)': {
        'url': 'https://www.gutenberg.org/cache/epub/13415/pg13415.txt',
        'start': 'THE LADY WITH THE DOG',
        'end': "A DOCTOR'S VISIT",
    },
    'Koteliansky & Cannon (1917)': {
        'url': 'https://www.gutenberg.org/cache/epub/27411/pg27411.txt',
        'start': 'THE LADY WITH THE TOY DOG',
        'end': 'GOUSSIEV',
    }
}

def load_text(url):
    with urllib.request.urlopen(url) as r:
        return r.read().decode('utf-8')

def extract_story(raw_text, start_marker, end_marker):

    start_pattern = rf'^\s*{re.escape(start_marker)}\s*$'

    matches = list(
        re.finditer(
            start_pattern,
            raw_text,
            flags=re.MULTILINE
        )
    )

    start_index = matches[1].start() # the first found title is in Contents, so I need the second one

    end_pattern = rf'^\s*{re.escape(end_marker)}\s*$'

    end_match = re.search(
        end_pattern,
        raw_text[start_index:],
        flags=re.MULTILINE
    )

    if end_match:
        end_index = start_index + end_match.start()
    else:
        end_index = len(raw_text)

    return raw_text[start_index:end_index].strip()

texts_raw = {}

for label, cfg in SOURCES.items():

    print(f'\n{label}')

    raw = load_text(cfg['url'])

    story = extract_story(
        raw,
        cfg['start'],
        cfg['end']
    )

    texts_raw[label] = story

    print(f'Length: {len(story):,} characters')


Garnett (1917)
Length: 37,013 characters

Koteliansky & Cannon (1917)
Length: 35,441 characters


### 2.2 Fetching from Wikisource

The final translation was sourced from Wikisource in the PDF format. It required specific preprocessing including the removal of page numbers, headers, the introduction to the story and various artefacts between the pages.

In [40]:
def download_pdf(url):
    response = requests.get(url)
    response.raise_for_status()
    return io.BytesIO(response.content)

def extract_pdf_text(pdf_file):
    reader = PdfReader(pdf_file)
    return [page.extract_text() or "" for page in reader.pages]


def clean_pages(pages):
    cleaned_pages = []

    for page in pages:
        # Deleting the numbers of pages
        page = re.sub(r'^\s*\d+\s*$', '', page, flags=re.MULTILINE)
        # Deleting the header
        page = re.sub(r"CHEKHOV\s*/\s*", "", page, flags=re.IGNORECASE)

        cleaned_pages.append(page.strip())

    return "\n".join(cleaned_pages)

In [41]:
def crop_story(text):
    # Removing the introduction
    start = re.search(r'^\s*I\s*$', text, flags=re.MULTILINE)
    end   = re.search(r'\[1899\]', text)

    if start and end:
        return text[start.start() : end.end()].strip()
    return text

def remove_print_markers(text):
    # Removing artefacts
    return re.sub(
        r'^\s*\d+\s+\d+\s+[\d\-]+\.ps\s+[\d/]+\s+[\d:]+\s+[AP]M\s+Page\s+\d+\s*$',
        '',
        text,
        flags=re.MULTILINE
    )

In [42]:
url = ("https://jerrywbrown.com/wp-content/uploads/2020/02/The-Lady-with-the-Dog-Chekhov.pdf")
label = "Litvinov (1973)"

pdf_file = download_pdf(url)
pages = extract_pdf_text(pdf_file)
story = clean_pages(pages)
story = crop_story(story)
story = remove_print_markers(story)


texts_raw[label] = story

print(f'\n{label}')

print(f"Story extracted: {len(story):,} characters")


Litvinov (1973)
Story extracted: 37,342 characters


### 2.3 Text cleaning

Extracted texts contain various excessive instances: end-of-line hyphenation, Roman numerals indicating chapters, extra whitespace. I applied the cleaning routine using regular expressions.

In [43]:
def clean_text(text):    
    # Re-join words split across lines by a hyphen
    text = re.sub(r'-\n\s*', '', text)
    # Removing Roman numerals indicating chapters
    text = re.sub(r'^\s*[IVXLC]+\s*$', '', text, flags=re.MULTILINE)
    # Replace newlines with spaces and collapse multiple spaces
    text = re.sub(r'\s{2,}', ' ', text.replace('\n', ' '))
    return text.strip()

texts_clean = {label: clean_text(t) for label, t in texts_raw.items()}

### 2.4 Saving

I saved the extracted texts as .txt files to allow for a manual preview before proceeding with the NLP pipeline.

In [44]:
PATHS = {
    'Garnett (1917)': 'garnett_lady_with_the_dog.txt',
    'Koteliansky & Cannon (1917)': 'koteliansky_lady_with_toy_dog.txt',
    'Litvinov (1973)': 'litvinov_lady_with_the_dog.txt'}

for label, path in PATHS.items():
    with open(path, 'w', encoding='utf-8') as f:
        f.write(texts_raw[label])

## 3. NLP Pipeline: Tokenization and Lemmatization <a class="anchor" id="pipeline"></a>
[Back to Table of Contents](#table-of-contents)

I applied the following NLTK pipeline:

**raw text → tokenization → POS tagging → lemmatization**

In [129]:
# Convert Penn Treebank POS tag to WordNet POS tag for lemmatization
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default fallback
    
def nlp_pipeline(text):
    # Step 1: tokenization
    tokens = word_tokenize(text)
    # Step 2: lowercase all tokens
    tokens_lower = [t.lower() for t in tokens]
    # Step 3: keep only alphabetic tokens (removes punctuation, numbers)
    tokens_alpha = [t for t in tokens_lower if re.fullmatch(r"[a-z]+(?:-[a-z]+)*", t)]
    # Step 4: remove stopwords
    tokens_clean = [t for t in tokens_alpha if t not in STOP_WORDS]
    # Step 5: POS tagging
    tagged_alpha = pos_tag(tokens_alpha)
    tagged = [(w, t) for w, t in tagged_alpha if w not in STOP_WORDS]
    # Step 6: lemmatization
    lemmas_alpha = [LEMMATIZER.lemmatize(word, get_wordnet_pos(tag)) 
                    for word, tag in tagged_alpha]
    lemmas = [l for l, w in zip(lemmas_alpha, tokens_alpha) if w not in STOP_WORDS]

    return {
        'tokens': tokens_lower,
        'tokens_alpha': tokens_alpha,
        'lemmas_alpha': lemmas_alpha,
        'tokens_clean': tokens_clean,
        'tagged': tagged,
        'lemmas': lemmas,
    }

processed = {label: nlp_pipeline(text) for label, text in texts_clean.items()}

# Basic statistics
rows = []
for label, data in processed.items():
    rows.append({
        'Translation': label,
        'Total tokens': len(data['tokens_alpha']),
        'Types (forms)': len(set(data['tokens_alpha'])),
        'Content lemmas': len(data['lemmas']),
        'Unique lemmas': len(set(data['lemmas'])),
    })

stats_df = pd.DataFrame(rows).set_index('Translation')
stats_df

,Total tokens,Types (forms),Content lemmas,Unique lemmas
Translation,,,,
Garnett (1917),6621,1472,2979,1180
Koteliansky & Cannon (1917),6364,1451,2883,1153
Litvinov (1973),6675,1572,3030,1245


## 4. Lexical Richness: Type-Token Ratio (TTR) <a class="anchor" id="ttr"></a>
[Back to Table of Contents](#table-of-contents)

I started my analysis with the TTR metrics estimating the lexical variety of the translations. 
Although TTR is strongly affected by the text length, a direct comparison is methodologically justified here since the three translations are comparable in size.

In [103]:
def compute_ttr(tokens_alpha):
    return len(set(tokens_alpha)) / len(tokens_alpha)

ttr_scores = {label: compute_ttr(data['tokens_alpha']) for label, data in processed.items()}

print("TTR scores:")
for label, score in ttr_scores.items():
    print(f"{label}: {score:.4f}")

TTR scores:
Garnett (1917): 0.2223
Koteliansky & Cannon (1917): 0.2280
Litvinov (1973): 0.2355


In [148]:
labels = list(ttr_scores.keys())
values = [ttr_scores[l] for l in labels]
colors = [PALETTE[l] for l in labels]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=labels,
        y=values,
        marker_color=colors,
        text=[f"{v:.4f}" for v in values],
        textposition='outside'
    )
)

fig.update_layout(
    title="Type-Token Ratio (TTR)<br>Three Translations of 'The Lady with the Dog'",
    xaxis_title="Translation",
    yaxis_title="TTR",
    yaxis=dict(
        range=[0, max(values) * 1.25]
    ),
    template="plotly_white",
    width=800,
    height=500
)
fig.show()

Fundamentally, TTR measures the range and diversity of the vocabulary implied by the translator. As shown in the bar chart, Litvinov's translation contains greater variety of different words. However, this metric alone is insufficient for drawing definitive conclusions.

## 5. Frequency Analysis: Top 20 Lemmas <a class="anchor" id="frequency"></a>
[Back to Table of Contents](#table-of-contents)

A frequency index lists lemmas with their number of occurrences. I computed both absolute and relative frequencies and produced a frequency table with pandas.

Comparing the top lemmas across translations reveals which concepts each translator foregrounds in their version of the story.

In [131]:
freq_tables = {}
for label, data in processed.items():
    freq = Counter(data['lemmas'])
    total = len(data['lemmas'])
    rows = [
        {'lemma': w, 'absolute_frequency': c, 'relative_frequency': round(c / total, 6)}
        for w, c in freq.most_common(30)
    ]
    freq_tables[label] = pd.DataFrame(rows)

for label, df in freq_tables.items():
    print(f"\n{label}. Top 20 lemmas")
    display(df.head(20))


Garnett (1917). Top 20 lemmas


,lemma,absolute_frequency,relative_frequency
0,one,40,0.013427
1,go,40,0.013427
2,gurov,35,0.011749
3,say,32,0.010742
4,come,32,0.010742
5,look,27,0.009063
6,anna,27,0.009063
7,sergeyevna,26,0.008728
8,life,25,0.008392
9,think,25,0.008392



Koteliansky & Cannon (1917). Top 20 lemmas


,lemma,absolute_frequency,relative_frequency
0,would,39,0.013528
1,go,39,0.013528
2,come,36,0.012487
3,gomov,32,0.011100
4,say,32,0.011100
5,one,30,0.010406
6,think,28,0.009712
7,woman,28,0.009712
8,anna,26,0.009018
9,sergueyevna,25,0.008672



Litvinov (1973). Top 20 lemmas


,lemma,absolute_frequency,relative_frequency
0,go,44,0.014521
1,one,35,0.011551
2,gurov,35,0.011551
3,say,29,0.009571
4,anna,28,0.009241
5,sergeyevna,26,0.008581
6,woman,25,0.008251
7,look,25,0.008251
8,life,25,0.008251
9,come,25,0.008251


The curious observation is that Koteliansky and Cannon translate the surname of one of the main characters as "Gomov", though the transliterated version is "Gurov".

## 6. Key Motif Frequencies <a class="anchor" id="motifs"></a>
[Back to Table of Contents](#table-of-contents)

Beyond the top-frequency lemmas, I distinguished specific motifs — words that are literarily significant in Chekhov's story from my point of view. I compared how often key terms appear in each translation, normalizing by total lemma count to make translations comparable.

The motifs below were selected based on close reading of the story:
- `love` — the leitmotif, the feeling, the core concept
- `sea` and `embankment` — the symbolic settings of the Yalta scenes
- `woman` — how Gurov perceives Anna 
- `secret` — the state that follows and embraces the relationship between Gurov and Anna
- `boredom` — the state of mind and soul that characters experienced before the life-changing meeting
- `silence` — one of the key modes of lovers' communication
- `memory` — the reflection, an essential part of the characters' inner lives
- `beautiful` - a crucial adjective closely attached to Anna
- `life` and `happiness` — subjects of philosophical reflections by the main characters

The word "набережная" appeared to be the most ambiguous one as all three translators used different English equivalents to render it, even though it is simply a specific location name:

- Garnett — sea-front
- Koteliansky and Cannon — quay
- Litvinov — promenade

While reading the translations and comparing them to the original, I identified several key motifs interpreted differently by each translator. These include:
- "скука" as bored (boring), dull (dulness)
- "тайно" as secret (secrecy), hidden, conceal (concealment)

In [134]:
MOTIF_GROUPS = {
    'love': ['love', 'loved', 'loving'],
    'sea': ['sea'],
    'life': ['life'],
    'woman': ['woman'],
    'secret': ['secret', 'secrecy', 'hidden', 'conceal', 'concealment'],
    'boredom': ['bored', 'boring', 'dull', 'dulness'], # скука
    'silence': ['silence', 'silent'], # молчание
    'embankment': ['sea-front', 'quay', 'promenade'], # набережная
    'memory': ['memory', 'remember', 'remembered', 'recall'], # воспоминание
    'beautiful': ['beautiful', 'lovely'], # красивая
    'happiness': ['happy', 'happiness', 'unhappy'],
}

motif_rows = []
for group_name, lemma_list in MOTIF_GROUPS.items():
    row = {'motif': group_name}
    for label, data in processed.items():
        freq  = Counter(data['lemmas'])
        total = len(data['lemmas'])
        count = sum(freq.get(lemma, 0) for lemma in lemma_list)
        row[label] = round(count / total * 10000, 2)
    motif_rows.append(row)

motif_df = pd.DataFrame(motif_rows).set_index('motif')
motif_df

,Garnett (1917),Koteliansky & Cannon (1917),Litvinov (1973)
motif,,,
love,63.78,69.37,59.41
sea,30.21,34.69,33.00
life,83.92,72.84,82.51
woman,80.56,97.12,82.51
secret,23.50,17.34,19.80
boredom,16.78,10.41,0.00
silence,26.85,27.75,19.80
embankment,13.43,13.87,13.20
memory,60.42,55.50,49.50


In [149]:
motif_names = list(motif_df.index)
translations  = list(motif_df.columns)

fig = go.Figure()

for label in translations:
    fig.add_trace(go.Bar(
        name = label,
        x = motif_names,
        y = motif_df[label].tolist(),
        marker = dict(color=PALETTE[label]),
    ))

fig.update_layout(
    barmode = 'group',
    title = "Key Motif Frequencies<br>Three Translations of 'The Lady with the Dog'",
    xaxis_title = "Motif",
    yaxis_title = "Frequency per 10,000 lemmas",
    legend_title= "Translation",
    xaxis = dict(tickangle=-40),
    template = "plotly_white",
    width=800,
    height=500
)
fig.show()

The bar chart makes it obvious that translators differently emphasized the core motifs of the story.

## 7. KWIC Concordances <a class="anchor" id="kwic"></a>
[Back to Table of Contents](#table-of-contents)

KWIC (Key-Word-In-Context) displays each occurrence of a word together with its surrounding context. For this reason, the analysis was performed on the `tokens_alpha` without stopword removal.

Using KWIC, I wanted to see how the same word is used in each translation, not just how often.

In [122]:
def kwic(tokens, lemmas, motif_groups, window=5, max_hits=5):
    counts = Counter(lemmas)

    for motif, variants in motif_groups.items():
        total = sum(counts[v] for v in variants)
        print(f"\nMotif: {motif.upper()}, total: {total}")

        for variant in variants:
            freq = counts[variant]
            if freq == 0:
                continue

            print(f"\n {variant} ({freq})")

            shown = 0
            for i, lemma in enumerate(lemmas):
                if lemma == variant:
                    left  = " ".join(tokens[max(0, i - window):i])
                    right = " ".join(tokens[i + 1:i + 1 + window])
                    print(f"{left} >> {tokens[i]} << {right}")
                    shown += 1
                    if shown >= max_hits:
                        break

for label, data in processed.items():
    print(f"KWIC analysis — {label}")
    kwic(
        data['tokens_alpha'],
        data['lemmas_alpha'],
        MOTIF_GROUPS,
        window=5,
        max_hits=5
    )
    print()

KWIC analysis — Garnett (1917)

Motif: LOVE, total: 19

 love (19)
thought of a swift fleeting >> love << affair a romance with an
of careless good-natured women who >> loved << cheerfully and were grateful to
women like his wife who >> loved << without any genuine feeling with
suggested that it was not >> love << nor passion but something more
beseech you she said i >> love << a pure honest life and

Motif: SEA, total: 9

 sea (9)
the strange light on the >> sea << the water was of a
to the roughness of the >> sea << the steamer arrived late after
a deathlike air but the >> sea << still broke noisily on the
church looked down at the >> sea << and were silent yalta was
monotonous hollow sound of the >> sea << rising up from below spoke

Motif: LIFE, total: 25

 life (25)
at first so agreeably diversifies >> life << and appears a light and
and he was eager for >> life << and everything seemed simple and
the first time in her >> life << she had been alone in
obstinate desire to snatch fr

## 8. Sentiment Trajectory <a class="anchor" id="sentiment"></a>
[Back to Table of Contents](#table-of-contents)

Deepening the analysis, I used VADER (Valence Aware Dictionary and sEntiment Reasoner) to identify the emotional tone of the text (positive, negative, neutral). It produces a score in the range `[-1, +1]`.

The sliding window technique was applied, dividing the text into overlapping segments of 500 words to plot the sentiment score.

Taking into considiration that VADER was originally designed for social media texts and its lexicon might not fully capture the specific emotional vocabulary of classical literature, this sentiment analysis is an exploratory experiment to evaluate the model's viability.

In [150]:
def sentiment_trajectory(text, window=500, step=100):

    words = text.split()
    positions = []
    scores = []

    for i in range(0, len(words) - window, step):
        seg = ' '.join(words[i : i + window])
        score = ANALYZER.polarity_scores(seg)['compound']
        positions.append(i / len(words))
        scores.append(score)
    return positions, scores

fig = go.Figure()

for label, text in texts_clean.items():
    pos, scores = sentiment_trajectory(text)
    fig.add_trace(go.Scatter(
        x = pos,
        y = scores,
        mode = 'lines',
        name = label,
        line = dict(color=PALETTE[label], width=2),
    ))

fig.add_hline(y=0, line_dash='dash', line_color='grey', opacity=0.6)

fig.update_layout(
    title = "Sentiment Trajectory<br>Three Translations of 'The Lady with the Dog'",
    xaxis_title = "Relative position in text",
    yaxis_title = "VADER score",
    legend_title= "Translation",
    template = "plotly_white",
    width=1000,
    height=600
)
fig.show()


The graph illustrates several parts of the texts where translators rendered diverging sentiment. In general, the text proves to be highly emotional, with almost no neutral vocabulary.

## 9. Character Analysis: Focalization <a class="anchor" id="character"></a>
[Back to Table of Contents](#table-of-contents)

Chekhov's story is told in the third person; however most of the story we see through Gurov's perspective, though Anna's interiority also matters. The ratio of masculine to feminine pronouns enables to indicate of which character each translation's narration dominates.

In [136]:
HE_FORMS = {'he', 'his', 'him', 'himself'}
SHE_FORMS = {'she', 'her', 'herself'}
GUROV_FORMS = {'gurov', 'gomov', 'dmitri', 'dmitry'}
ANNA_FORMS  = {'anna'}

def character_profile(tokens):

    total = len(tokens)
    he = sum(1 for t in tokens if t in HE_FORMS)
    she = sum(1 for t in tokens if t in SHE_FORMS)
    gurov = sum(1 for t in tokens if t in GUROV_FORMS)
    anna = sum(1 for t in tokens if t in ANNA_FORMS)
    return {
        'he/him/his': round(he / total * 1000, 2),
        'she/her/hers': round(she / total * 1000, 2),
        'Gurov (name)': round(gurov / total * 1000, 2),
        'Anna (name)': round(anna / total * 1000, 2),
        'she:he ratio': round(she / he, 3) if he > 0 else None,
    }

pron_data = {
    label: character_profile(data['tokens_alpha'])
    for label, data in processed.items()
}

pron_df = pd.DataFrame(pron_data).T
pron_df

,he/him/his,she/her/hers,Gurov (name),Anna (name),she:he ratio
Garnett (1917),53.16,28.70,5.89,4.08,0.540
Koteliansky & Cannon (1917),52.64,32.53,5.03,4.09,0.618
Litvinov (1973),50.79,29.66,5.84,4.19,0.584


## 10. Hapax Legomena <a class="anchor" id="hapax"></a>
[Back to Table of Contents](#table-of-contents)

Hapax legomena reveals the words that appear only once in a text. A high hapax count relates to more broad and varied lexicon. The author introduces many unique words rather than relies on a repeated vocabulary.

In the context of translation as modelling: hapaxes are the translator's most individual choices, words that appear in no other sentence and therefore reflect a unique creative decision.

In [137]:
hapax_rows = []

for label, data in processed.items():

    freq = Counter(data['lemmas'])
    hapax = sorted([w for w, c in freq.items() if c == 1])
    total = len(data['lemmas'])

    hapax_rows.append({
        'Translation': label,
        'Hapax count': len(hapax),
        'Total lemmas': total,
        'Hapax %': round(len(hapax) / total * 100, 2),
        'Sample': ', '.join(hapax[:8]),
    })

hapax_df = pd.DataFrame(hapax_rows).set_index('Translation')
hapax_df

,Hapax count,Total lemmas,Hapax %,Sample
Translation,,,,
Garnett (1917),706,2979,23.70,"able, accidental, admire, adore, affair, affec..."
Koteliansky & Cannon (1917),693,2883,24.04,"abrupt, absently, accompany, accursed, ache, a..."
Litvinov (1973),756,3030,24.95,"abrupt, accent, accidental, account, accursed,..."


This metric aligns closely with my experience of reading the translations as a non-native English speaker. Litvinov employs broader variety of synonyms and adjectives, while Garnett's text is more straightforward and simple to read.

## 11. Syntactic Profile: Sentence Length and POS Density <a class="anchor" id="syntax"></a>
[Back to Table of Contents](#table-of-contents)

Differents translators can share a similar vocabulary, and yet produce texts that feel completely different because of the sentence structure.

This section adds a syntactic dimension through two complementary metrics:

1. **Sentence length distribution** — the average, median, and spread of sentence lengths in tokens.
2. **POS density** — the proportion of adjectives, adverbs, verbs, and nouns among all tokens.

In [142]:
def sentence_stats(text):
    sentences = sent_tokenize(text)
    lengths = [len(word_tokenize(s)) for s in sentences]
    n = len(lengths)
    mean = sum(lengths) / n
    sorted_l = sorted(lengths)
    median = sorted_l[n // 2]
    std = math.sqrt(sum((x - mean) ** 2 for x in lengths) / n)
    return {
        'Sentence count': n,
        'Mean length': round(mean, 1),
        'Median length': median,
        'Std': round(std, 1),
        'Max length': max(lengths),
    }

sent_rows = []
for label, text in texts_clean.items():
    row = {'Translation': label}
    row.update(sentence_stats(text))
    sent_rows.append(row)

sent_df = pd.DataFrame(sent_rows).set_index('Translation')
print("Sentence length statistics:")
sent_df

Sentence length statistics:


,Sentence count,Mean length,Median length,Std,Max length
Translation,,,,,
Garnett (1917),337,23.4,18,19.9,153
Koteliansky & Cannon (1917),350,21.6,15,20.0,153
Litvinov (1973),358,22.6,19,21.3,181


The table illustrates that Litvinov's text is syntatically richer: it is not only contains more sentances, but also have a greater averafe length.

In [141]:
def pos_density(tagged):
    total = len(tagged)
    counts = Counter(tag for _, tag in tagged)

    adj = sum(v for t, v in counts.items() if t.startswith('JJ'))
    adv = sum(v for t, v in counts.items() if t.startswith('RB'))
    verb = sum(v for t, v in counts.items() if t.startswith('VB'))
    noun = sum(v for t, v in counts.items() if t.startswith('NN'))
    return {
        'Adjectives %': round(adj / total * 100, 2),
        'Adverbs %': round(adv / total * 100, 2),
        'Verbs %': round(verb / total * 100, 2),
        'Nouns %': round(noun / total * 100, 2),
    }

pos_rows = []
for label, data in processed.items():
    row = {'Translation': label}
    row.update(pos_density(data['tagged']))
    pos_rows.append(row)

pos_df = pd.DataFrame(pos_rows).set_index('Translation')
print("POS density (% of all tokens):")
pos_df

POS density (% of all tokens):


,Adjectives %,Adverbs %,Verbs %,Nouns %
Translation,,,,
Garnett (1917),14.97,7.69,29.71,40.45
Koteliansky & Cannon (1917),14.53,8.29,29.93,40.20
Litvinov (1973),14.09,7.95,29.21,41.98


The usage of different parts of speech seems remarkably similar across all three translations.

In [151]:
categories = ['Adjectives %', 'Adverbs %', 'Verbs %', 'Nouns %']
fig = go.Figure()

for label in pos_df.index:
    fig.add_trace(go.Bar(
        name = label,
        x = categories,
        y = [pos_df.loc[label, c] for c in categories],
        marker = dict(color=PALETTE[label]),
    ))

fig.update_layout(
    barmode = 'group',
    title = "POS Density<br>Three Translations of 'The Lady with the Dog'",
    xaxis_title = "Part of Speech",
    yaxis_title = "% of tokens",
    legend_title= "Translation",
    template = "plotly_white",
    width=800,
    height=500
)
fig.show()

## 12. Discussion <a class="anchor" id="discussion"></a>
[Back to Table of Contents](#table-of-contents)

### What this analysis reveals

The computational comparison across the three translations provides evidence for several interpretive differences:

1. **TTR differences** demonstrate that the translations do not reproduce the same lexical variety. Even though they are translating the same text, each translator's word choices create a measurably different lexical model of the story. 

Litvinov's higher TTR score indicates that she opted for the richer vocabulary - a stylistic choice that reflects both her conscious effort to mirror the Chekov's precision and her historical distance from the other two translators (her work is more recent). In contrast, Garnett's lower TTR highlights fluency and simpicity in her approach to translation. The results align with the hapax score and syntactic profile.

2. **Motif frequency differences** show that translators do not emphasize the story's key concepts equally.

The lemma `woman` appears more frequently in Koteliansky and Cannon’s translation with approximated score 0.01 compared to 0.008 in the other two. This may indicate an «Anna-centered» perspective adopted by these authors, which I will examine further during the character analysis.

Regarding the lemma `love`, it takes almost the last place in Litvinov’s top 20, whereas it receives higher frequency scores in other two texts. Given that «The Lady with the Dog» is fundamentally a love story, this suggests Litvinov’s shift toward more implicit and metaphorical ways of transfering the love motif, while the other translators name this feeling directly.

Furthermore, the motif cluster for `boredom` is absent in Litvinov’s translation, although it appears in the  others. It is worth noting that here by boredom I mean the state the characters were experiencing before their life-changing meeting. 

Finally, Garnett seems to pay more attention to the philosophical motifs of `beauty` and `memory` than her counterparts. She also emphasizes the `secrecy` surrounding Gurov and Anna’s relationship.

3. **KWIC concordances** illustrate that even when the same word appears in multiple translations, the surrounding context may differ, reflecting different narrative choices.

One of the most vivid examples is the occurrence of the word `secret` which differs significantly across the parts of the passage it is used in by the translators.

4. **Sentiment trajectories** reveal both divergence and convergence of the emotional tone across different translations.

Convergence in the Yalta section suggests that the beginning of the affair is a structural feature that all three translators preserve. On the contrary, divergences in the Moscow parts are more likely to be modelling decisions. Specifically, Koteliansky and Cannon produce more frequently negative perception than the other two translators.

5. **Character ratios** answer whether the translator shifts his emphasis toward Gurov's or Anna's perspective in the ambiguous parts. 

Thus, Koteliansky and Cannon's higher she:he ratio signals that they grant more textual space for Anna, while Garett's lower rate foregrounds Gurov as the narrative center.

### What this analysis does not reveal

Following the idea that modelling is choosing, I should also name the limits of this approach:

- **VADER's trainig data:** as I have already mentioned, the fact that VADER was trained on social media texts may limit its ability to distinguish some nuances of the traditional literature.
- **The absent original:** I was comparing three English models but did not have the Russian original in the analysis. I cannot say which translation is closer to Chekhov, only that they differ from each other.

## 13. Conclusion <a class="anchor" id="conclusion"></a>
[Back to Table of Contents](#table-of-contents)

In this notebook the NLP pipeline was applied to a research question: **can computational text analysis measure the interpretive differences between translations of the same literary text?**

The answer is yes, and all eight metrics confirm that the three translations are not equivalent textual models of the same story.

here are three ideas run through the entire analysis:

1. **Modelling is choosing.** Every step in this notebook required a decision: which motifs to track, what window size for sentiment, how to extract the story from the Gutenberg file. The numbers are the consequence of those choices.

2. **Visualization is interpretation.** Each bar chart and trajectory plot is already an argument: it foregrounds some differences and hides others.

3. **The toolkit is not universal.** NLTK and VADER were designed for contemporary English. Using them on the 20th century prose is a cross-domain transfer.

To sum up, these three translations represent three different models of the same text. The computational analysis performed here allowed us to reveal those differences and make them evident, moving beyond the subjective impressions gained through the close reading.

*See `README.md` in the repository for the full workflow documentation, data sources, and methodological notes.*